In [ ]:
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
from scipy.optimize import root_scalar

In [ ]:
def compute_bayes_error_mps(X, Y, k=50, ref_size=1000000, sample_size=100000, batch_size=1000):
    """
    Memory-safe k-NN Bayes Error Rate R* on Apple Silicon MPS.
    
    Parameters:
    - X: np.ndarray (N, D) float32 features
    - Y: np.ndarray (N,) binary labels
    - k: Number of nearest neighbors
    - ref_size: Size of index reference set (1M points is statistically sufficient)
    - sample_size: Number of query points to evaluate
    - batch_size: Small batch size to keep MPS memory under 4GB
    """
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Running memory-safe k-NN on device: {device}")

    N, D = X.shape
    
    # 1. Subsample reference pool (X_ref) to keep distance matrix small
    if N > ref_size:
        ref_idx = np.random.choice(N, size=ref_size, replace=False)
        X_ref = X[ref_idx]
        Y_ref = Y[ref_idx]
    else:
        X_ref = X
        Y_ref = Y
        ref_size = N

    # 2. Subsample query points (X_query)
    if N > sample_size:
        query_idx = np.random.choice(N, size=sample_size, replace=False)
        X_query = X[query_idx]
    else:
        X_query = X

    # Move reference data to MPS GPU
    X_ref_t = torch.from_numpy(X_ref.astype(np.float32)).to(device)
    Y_ref_t = torch.from_numpy(Y_ref.astype(np.int32)).to(device)
    X_query_t = torch.from_numpy(X_query.astype(np.float32)).to(device)

    # Pre-compute reference squared norms: (1, ref_size)
    X_ref_norms = torch.sum(X_ref_t ** 2, dim=1, keepdim=True).T

    bayes_errors = []

    print(f"Querying {k}-NN: {sample_size:,} queries against {ref_size:,} reference vectors...")

    for i in tqdm(range(0, sample_size, batch_size)):
        X_batch = X_query_t[i:i+batch_size]  # (batch_size, D)
        batch_norms = torch.sum(X_batch ** 2, dim=1, keepdim=True)  # (batch_size, 1)

        # Distances shape: (batch_size, ref_size) -> 1000 x 1,000,000 = 4 GB allocation max
        dists = batch_norms + X_ref_norms - 2 * torch.mm(X_batch, X_ref_t.T)

        # Top-k nearest neighbors
        _, topk_indices = torch.topk(dists, k=k, dim=1, largest=False)

        # Local P(Y=1 | X)
        neighbor_labels = Y_ref_t[topk_indices].float()
        p_hat = torch.mean(neighbor_labels, dim=1)

        # Local Bayes Error = min(P(Y=1|X), P(Y=0|X))
        local_bayes_error = torch.minimum(p_hat, 1.0 - p_hat)
        bayes_errors.append(local_bayes_error.cpu())

    bayes_errors = torch.cat(bayes_errors).numpy()
    global_bayes_error = float(np.mean(bayes_errors))
    max_accuracy = (1.0 - global_bayes_error) * 100.0

    print(f"\n==============================================")
    print(f"--- Empirical Bayes Floor Results (Apple MPS) ---")
    print(f"K: {k}")
    print(f"Irreducible Error Floor (R*): {global_bayes_error * 100:.2f}%")
    print(f"Max Possible Classifier Accuracy: {max_accuracy:.2f}%")
    print(f"==============================================\n")

    return global_bayes_error, max_accuracy


In [ ]:
def binary_entropy(p):
    """Calculates Binary Entropy H_b(p) in bits."""
    p = np.clip(p, 1e-12, 1.0 - 1e-12)
    return -p * np.log2(p) - (1.0 - p) * np.log2(1.0 - p)

def compute_fano_entropy_bound_pytorch(X, Y, n_clusters=1000, max_iters=20, sample_size=2000000):
    """
    Computes Conditional Entropy H(Y|X) and Fano's Bound using memory-safe PyTorch K-Means.
    
    Parameters:
    - X: np.ndarray (N, D) float32 features
    - Y: np.ndarray (N,) binary labels {0, 1}
    - n_clusters: Number of micro-clusters (1,000 to 2,000)
    - sample_size: Subsample used to train centroids (2M is plenty to learn centroids)
    """
    N, D = X.shape
    
    # 1. Subsample to fit centroids safely in memory
    if N > sample_size:
        idx = np.random.choice(N, size=sample_size, replace=False)
        X_train = torch.from_numpy(X[idx].astype(np.float32))
    else:
        X_train = torch.from_numpy(X.astype(np.float32))

    print(f"Training {n_clusters} centroids using PyTorch on 2M sample...")
    
    # Randomly initialize centroids
    perm = torch.randperm(X_train.size(0))
    centroids = X_train[perm[:n_clusters]].clone()

    # Mini-batch K-Means training loop (runs fast on CPU/MPS)
    for iteration in range(max_iters):
        # Distance to centroids: (N, n_clusters)
        dists = torch.cdist(X_train, centroids)
        assignments = torch.argmin(dists, dim=1)
        
        # Update centroids
        new_centroids = torch.zeros_like(centroids)
        for k in range(n_clusters):
            mask = (assignments == k)
            if mask.sum() > 0:
                new_centroids[k] = X_train[mask].mean(dim=0)
            else:
                new_centroids[k] = centroids[k]
        
        center_shift = torch.norm(new_centroids - centroids)
        centroids = new_centroids
        if center_shift < 1e-4:
            break

    print("Assigning full 20M dataset to centroids in safe chunks...")
    
    # 2. Assign all 20M points in safe chunks of 100,000
    chunk_size = 100000
    all_assignments = []
    
    X_tensor = torch.from_numpy(X.astype(np.float32))
    for i in tqdm(range(0, N, chunk_size)):
        X_chunk = X_tensor[i:i+chunk_size]
        dists = torch.cdist(X_chunk, centroids)
        assign = torch.argmin(dists, dim=1).numpy()
        all_assignments.append(assign)
        
    cluster_assignments = np.concatenate(all_assignments)

    # 3. Compute Conditional Entropy H(Y|X)
    print("Computing Conditional Entropy H(Y|X)...")
    cluster_counts = np.bincount(cluster_assignments, minlength=n_clusters)
    positive_counts = np.bincount(cluster_assignments, weights=Y, minlength=n_clusters)

    valid_mask = cluster_counts > 0
    p_ck = cluster_counts[valid_mask] / N
    p_y1_given_ck = positive_counts[valid_mask] / cluster_counts[valid_mask]

    h_local = binary_entropy(p_y1_given_ck)
    h_y_given_x = float(np.sum(p_ck * h_local))

    # 4. Fano's Inequality Solve
    target_func = lambda pe: binary_entropy(pe) - h_y_given_x
    sol = root_scalar(target_func, bracket=[1e-7, 0.5], method='brentq')
    p_e_lower_bound = sol.root
    fano_max_accuracy = (1.0 - p_e_lower_bound) * 100.0

    print(f"\n==============================================")
    print(f"--- Fano's Information Bound Results ---")
    print(f"Number of Clusters: {n_clusters}")
    print(f"Conditional Entropy H(Y|X): {h_y_given_x:.4f} bits")
    print(f"Fano Error Lower Bound (P_e): {p_e_lower_bound * 100:.2f}%")
    print(f"Max Possible Accuracy (Information Floor): {fano_max_accuracy:.2f}%")
    print(f"==============================================\n")

    return h_y_given_x, p_e_lower_bound, fano_max_accuracy

In [14]:
df = pd.read_parquet('data/v80_new_features/final_model_data_the_rescaled.parquet')

In [15]:
X_test = df[['Bx', 'By', 'Bz', 'Bx_lag_1', 'Bx_lag_2', 'By_lag_1',
             'By_lag_2', 'Bz_lag_1', 'Bz_lag_2', 'Bx_conditional_vol',
             'By_conditional_vol', 'Bz_conditional_vol', 'LIM_scale_2_log',
             'LIM_scale_4_log', 'LIM_scale_8_log', 'LIM_scale_16_log',
             'LIM_scale_32_log', 'LIM_scale_64_log']].values

y_test = df['Event_label_80'].values

In [16]:
del df

In [6]:
# Run command:
R_star, max_acc = compute_bayes_error_mps(X_test, y_test, k=100)

Running memory-safe k-NN on device: mps
Querying 100-NN: 100,000 queries against 1,000,000 reference vectors...


100%|██████████| 100/100 [03:44<00:00,  2.24s/it]


--- Empirical Bayes Floor Results (Apple MPS) ---
K: 100
Irreducible Error Floor (R*): 20.33%
Max Possible Classifier Accuracy: 79.67%



In [7]:
# h_yx, min_err, max_acc = compute_fano_entropy_bound_pytorch(X_test, y_test, n_clusters=4000)